### Cleaning card transactions

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bankof420.bank_cards_silver;
CREATE SCHEMA IF NOT EXISTS bankof420.bank_gold;
CREATE SCHEMA IF NOT EXISTS bankof420.bank_core_silver;
CREATE SCHEMA IF NOT EXISTS bankof420.bank_cards_silver;
CREATE SCHEMA IF NOT EXISTS bankof420.bank_loans_silver;




In [0]:
from pyspark.sql.functions import col

silver_card_txn = (
    spark.table("bankof420.bank_cards.card_transactions")
    .filter(col("amount") > 0)
    .dropDuplicates(["card_txn_id"])
)

silver_card_txn.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bankof420.bank_cards_silver.card_transactions")


### Cleaning account

In [0]:
from pyspark.sql.functions import col

silver_accounts = (
    spark.table("bankof420.bank_core.accounts")
    .filter(col("balance") >= 0)
)

silver_accounts.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bankof420.bank_core_silver.accounts")


In [0]:
silver_cards = (
    spark.table("bankof420.bank_cards.cards")
    .dropDuplicates(["card_id"])
)

silver_cards.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bankof420.bank_cards_silver.cards")


In [0]:
silver_loans = (
    spark.table("bankof420.bank_loans.loans")
    .dropDuplicates(["loan_id"])
)

silver_loans.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bankof420.bank_loans_silver.loans")


inncrementle load 

In [0]:
from pyspark.sql.functions import max

last_txn_date = spark.sql("""
SELECT COALESCE(MAX(txn_date), '1900-01-01')
FROM bankof420.bank_cards_silver.card_transactions
""").collect()[0][0]

incremental_df = (
    spark.table("bankof420.bank_cards.card_transactions")
    .filter(f"txn_date > '{last_txn_date}'")
)

incremental_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("bankof420.bank_cards_silver.card_transactions")
